# BrainDecoding — Google Colab GPU 秒级解包、混合精度训练与验证集可视化评估工作流

本 Notebook 运行在 **Google Colab GPU 实例**（如 L4 / A100 / T4）上：
1. 自动创建并检查 Google Drive 标准目录（`/content/drive/MyDrive/BrainDecoding/{archives,runs,data,models}`）；
2. 从 Google Drive 秒级解包单文件缓存归档 `archives/features_<modality>.tar` 到本地高速盘 `/content/local_cache`；
3. 在本地 `/content/studies_links` 建立与 CPU 预热阶段完全一致的软链接桥（确保缓存签名 100% 命中）；
4. 通过 `--config` 直接读取 `configs/training/*.yaml` 配置文件（支持 **阶段 1: 词级脑编码器**、**阶段 2.1: 词级脑编码器 + 16 层全局上下文 Transformer**、**阶段 2.2: 词级脑编码器 + 16 层局部窗口滚动修订**）；
5. 支持通过 `EXP_TAG`（输出目录标签）与 `WORD_EXP_TAG`（阶段 2 热启动所引用的阶段 1 实验标签）灵活组合不同超参数实验；
6. 训练完成后一键执行**验证集 (`val`) 全面评估**，并在 Notebook 内直观绘制多词表规模检索、分受试者表现、训练收敛曲线与跨阶段对比图表。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、自动创建目录与配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 直接选择 configs/training/ 下的清晰 YAML 配置文件（超参数已针对 L4 GPU 调优）：
#   - 'configs/training/pallier2025_word_encoder.yaml'          (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'   (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'      (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'          (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'   (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'      (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
CONFIG_FILE = 'configs/training/pallier2025_context_transformer.yaml'
MODALITY = 'meg'                      # 对应归档标识: 'meg' (pallier2025) 或 'eeg' (chineseeeg2)
EXP_TAG = 'stage1_freeze_bs256_stage2_lr1e-4_bs128'              # 当前训练输出实验标签（输出为 runs/<dataset>/<EXP_TAG>/<stage>/...；留空 '' 则为 runs/<dataset>/<stage>/...）
WORD_EXP_TAG = 'lr4e-4_bs256'                     # 阶段 2 (2.1/2.2) 热启动加载哪个实验标签下的阶段 1 (word) 权重（留空 '' 默认与 EXP_TAG 相同；填 'default' 加载无标签根目录 word）
SEED = 0                              # 随机种子: 0, 1, 2
SUBSET_ARGS = '--subject all'         # 或 '--num-subjects 2' / '--subjects sub-01,sub-02'
# ========================================================

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

# 统一通过环境变量驱动所有路径（与 CPU 预热完全对齐）
os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('CONFIG_FILE               =', CONFIG_FILE)
print('BRAINDECODING_DATA_DIR    =', os.environ['BRAINDECODING_DATA_DIR'])
print('BRAINDECODING_CACHE_DIR   =', os.environ['BRAINDECODING_CACHE_DIR'])
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_EXP_TAG     =', os.environ['BRAINDECODING_EXP_TAG'] or '(default)')
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 防崩版初始化虚拟环境（锁定系统 PyTorch/CUDA 版本）+ 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 将 Google Drive 上的单一 .tar 缓存包秒级解包回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 启动 GPU 训练（通过 --config 指定 YAML 配置文件 + 实时 tqdm 进度条 + 自动从 last.ckpt 断点续训）
# 输出路径：{DRIVE_RESULTS_DIR}/<dataset>/<EXP_TAG>/<stage>/<subset_tag>/seed-000
# 若为阶段 2.1/2.2 且设置了 WORD_EXP_TAG，会自动从 {DRIVE_RESULTS_DIR}/<dataset>/<WORD_EXP_TAG>/word/.../best.ckpt 加载阶段 1 权重
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding train \
    --config {CONFIG_FILE} \
    --seed {SEED} \
    --require-cache \
    --resume auto \
    {SUBSET_ARGS}

In [ ]:
# 6. 验证集 (val) 全面评估与可视化图表展示（多词表规模检索 + 分受试者精度 + 收敛曲线 + 跨阶段对比）
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --config {CONFIG_FILE} \
    --seed {SEED} \
    --split val \
    --require-cache \
    {SUBSET_ARGS}

from IPython.display import Image, display

eval_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*_val*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
shown = set()
for png_path in eval_pngs:
    if EXP_TAG and EXP_TAG not in png_path.parts:
        continue
    if png_path.name in shown:
        continue
    shown.add(png_path.name)
    print(f'\n[图表预览] {png_path}')
    display(Image(filename=str(png_path)))